# Customer Churn Intelligence — Class Imbalance Experiments

## Objective

Compare imbalance-handling strategies for **Logistic Regression** and **XGBoost** on train/validation data only. The **final test set is not used**.

**Stage:** Step 14 — Imbalance experiments (no threshold tuning, calibration, SHAP, or test-set evaluation).

### Strategy overview

| # | Strategy | Description |
|---|----------|-------------|
| 1 | **Original** | No resampling; no class weights (`scale_pos_weight=1` for XGBoost) |
| 2 | **Class weighting** | `class_weight='balanced'` (LR) or `scale_pos_weight=neg/pos` (XGBoost) |
| 3 | **RandomUnderSampler** | Randomly downsample majority class on **training data only** (after preprocessing) |
| 4 | **SMOTENC** | Synthetic oversampling for **mixed numeric/categorical** raw features (before preprocessing) |

### Why SMOTENC (not vanilla SMOTE)?
Vanilla SMOTE treats all features as continuous and can create **invalid categorical values**.  
**SMOTENC** knows which columns are categorical (15 object columns + numeric kept separate) and generates synthetically balanced training rows **without corrupting category levels**. It is applied **only inside the training pipeline** before one-hot encoding.

### Imbalance handling concepts
- **Class weights** — simple first approach; reweights the loss without changing data; no synthetic rows; no row loss.
- **Under-sampling** — balances classes but **discards majority-class examples** that may contain useful patterns.
- **Oversampling (SMOTENC)** — can improve minority-class signal but may create **synthetic points** that do not represent real customers.
- **Validation/test must never be resampled** — we need honest evaluation on the **real churn rate (~26.5%)**.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
IMBALANCE_COMPARISON_PATH = REPORTS_DIR / "imbalance_comparison.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_separation import FEATURE_COLS, NUMERIC_FEATURE_COLS
from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42

## 1. Load Train / Validation (test excluded)

In [ ]:
split = load_split_from_manifest()

X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = neg_count / pos_count

# Categorical column indices in raw FEATURE_COLS order (for SMOTENC)
categorical_indices = list(range(len(NUMERIC_FEATURE_COLS), len(FEATURE_COLS)))

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")
print(f"Train Yes rate: {y_train.mean():.2%} | Validation Yes rate: {y_val.mean():.2%}")
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.4f}")
print(f"SMOTENC categorical indices: {categorical_indices}")

## 2. Build Leakage-Safe Imbalanced Pipelines

In [ ]:
def build_imbalanced_pipeline(
    model,
    strategy: str,
) -> ImbPipeline:
    """Build pipeline with resampling applied only during fit on training data."""
    preprocessor = build_preprocessor()

    if strategy == "Original":
        steps = [("preprocessor", preprocessor), ("model", model)]
    elif strategy == "Class weighting":
        steps = [("preprocessor", preprocessor), ("model", model)]
    elif strategy == "RandomUnderSampler":
        # Undersample after preprocessing — validation still uses transform() only
        steps = [
            ("preprocessor", preprocessor),
            ("undersample", RandomUnderSampler(random_state=RANDOM_STATE)),
            ("model", model),
        ]
    elif strategy == "SMOTENC":
        # SMOTENC on raw mixed features BEFORE one-hot encoding
        steps = [
            ("smotenc", SMOTENC(categorical_features=categorical_indices, random_state=RANDOM_STATE)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return ImbPipeline(steps=steps)


def make_logistic_regression(strategy: str) -> LogisticRegression:
    if strategy == "Class weighting":
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)


def make_xgboost(strategy: str) -> XGBClassifier:
    spw = scale_pos_weight if strategy == "Class weighting" else 1.0
    return XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    )

## 3. Run Experiments

In [ ]:
STRATEGIES = ["Original", "Class weighting", "RandomUnderSampler", "SMOTENC"]
MODEL_BUILDERS = {
    "LogisticRegression": make_logistic_regression,
    "XGBoost": make_xgboost,
}

experiment_rows = []
confusion_matrices = {}

for model_name, builder in MODEL_BUILDERS.items():
    for strategy in STRATEGIES:
        pipe = build_imbalanced_pipeline(builder(strategy), strategy)
        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_val)
        y_proba = pipe.predict_proba(X_val)[:, 1]

        key = f"{model_name} | {strategy}"
        confusion_matrices[key] = confusion_matrix(y_val, y_pred)

        experiment_rows.append(
            {
                "Model": model_name,
                "Imbalance_Strategy": strategy,
                "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
                "Recall": round(recall_score(y_val, y_pred), 4),
                "F1": round(f1_score(y_val, y_pred), 4),
                "ROC_AUC": round(roc_auc_score(y_val, y_proba), 4),
                "PR_AUC": round(average_precision_score(y_val, y_proba), 4),
                "Predicted_Churners": int(y_pred.sum()),
            }
        )

results_df = pd.DataFrame(experiment_rows)
results_df = results_df.sort_values(["Model", "PR_AUC"], ascending=[True, False])
results_df

In [ ]:
for key, cm in confusion_matrices.items():
    cm_df = pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"],
    )
    print(f"\nConfusion Matrix — {key}")
    display(cm_df)

## 4. Comparison Table

In [ ]:
comparison_df = results_df[
    ["Model", "Imbalance_Strategy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
].copy()

comparison_df.to_csv(IMBALANCE_COMPARISON_PATH, index=False)
print(f"Saved: {IMBALANCE_COMPARISON_PATH}")
comparison_df.sort_values("PR_AUC", ascending=False)

## 5. Findings

In [ ]:
best_pr = comparison_df.sort_values("PR_AUC", ascending=False).iloc[0]
best_f1 = comparison_df.sort_values("F1", ascending=False).iloc[0]

print(f"Best strategy by PR-AUC: {best_pr['Model']} + {best_pr['Imbalance_Strategy']} (PR-AUC={best_pr['PR_AUC']})")
print(f"Best strategy by F1: {best_f1['Model']} + {best_f1['Imbalance_Strategy']} (F1={best_f1['F1']})")

for model in comparison_df["Model"].unique():
    subset = comparison_df[comparison_df["Model"] == model]
    cw = subset[subset["Imbalance_Strategy"] == "Class weighting"]["PR_AUC"].iloc[0]
    best_resample = subset[subset["Imbalance_Strategy"].isin(["RandomUnderSampler", "SMOTENC"])]["PR_AUC"].max()
    improved = best_resample > cw
    print(f"\n{model}: class weighting PR-AUC={cw}; best resampling PR-AUC={best_resample}; resampling beats weights={improved}")

## Summary

- **8 experiments** (2 models × 4 strategies), all resampling confined to training `fit()`
- **Validation distribution unchanged** (~26.5% churn)
- **Test set:** not used

**Next step (not performed here):** Consolidated model comparison or probability calibration.